# EKS MCP 서버를 AgentCore Gateway에 연결하기

이 실습에서는 프라이빗 VPC 내부의 Amazon EKS에 [FastMCP](https://github.com/jlowin/fastmcp) 서버 두 개를 배포합니다. 단일 내부 NLB 뒤의 [NGINX Ingress Controller](https://kubernetes.github.io/ingress-nginx/)를 통해 서버를 제공한 다음, 관리형 VPC 송신을 사용하여 [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)에 연결합니다.

MCP 서버에는 VPC와 연결된 Route 53 프라이빗 호스팅 영역의 **프라이빗 도메인**을 통해 접근할 수 있습니다. VPC에서 **Private DNS**를 활성화하면(기본값), AgentCore Gateway의 관리형 Resource Gateway가 VPC의 DNS 해석기를 통해 도메인을 해석하므로 `routingDomain` 우회 방법이 필요하지 않습니다. NGINX는 경로 기반 라우팅을 수행하므로 단일 NLB가 두 MCP 서버(`/mcp-server/mcp` 및 `/stock-mcp/mcp`)를 모두 제공합니다.

VPC 송신, 인증서 요구 사항 및 Private DNS에 관한 배경 정보는 [프로젝트 README](../README.md), [Managed VPC Resource README 문서](../01-managed-vpc-resource/README.md), [사전 요구 사항](../00-prerequisites/)을 참조하세요.

![아키텍처](./images/eks-mcp.png)

## 사전 요구 사항

- [실습 0](../00-prerequisites/00-vpc-gateway-setup.ipynb) 완료(VPC + AgentCore Gateway 배포)
- Docker 실행 중(CDK 컨테이너 이미지 빌드에 필요)
- NLB에서 TLS를 종료하기 위한 [ACM 퍼블릭 인증서](../00-prerequisites/create-acm-public-certificate.md)

## 1단계: 종속성 설치 및 라이브러리 가져오기

In [ ]:
import os
from pathlib import Path

# 프로젝트 루트로 이동
cwd = Path.cwd()
while cwd != cwd.parent:
    if (cwd / "cdk.json").exists():
        break
    cwd = cwd.parent
os.chdir(cwd)
print(f"Working directory: {os.getcwd()}")

!pip install --force-reinstall -q -r requirements.txt

In [ ]:
import json
import os
import time

import boto3
from utils.utils import get_token

# 실습 0에서 변수 복원
%store -r ACCOUNT_A_ID
%store -r ACCOUNT_A_PROFILE
%store -r GATEWAY_ID
%store -r GATEWAY_URL
%store -r USER_POOL_ID
%store -r USER_POOL_CLIENT_ID
%store -r TOKEN_ENDPOINT_URL
%store -r OAUTH_SCOPES
%store -r VPC_USW2_ID
%store -r VPC_USW2_PRIVATE_SUBNETS

os.environ["ACCOUNT_A_ID"] = ACCOUNT_A_ID

REGION = "us-west-2"
session = boto3.Session(profile_name=ACCOUNT_A_PROFILE, region_name=REGION)
agentcore = session.client("bedrock-agentcore-control")

# Cognito 클라이언트 보안 암호 가져오기
cognito = session.client("cognito-idp")
client_desc = cognito.describe_user_pool_client(UserPoolId=USER_POOL_ID, ClientId=USER_POOL_CLIENT_ID)
CLIENT_SECRET = client_desc["UserPoolClient"]["ClientSecret"]

print(f"Account:    {ACCOUNT_A_ID}")
print(f"Region:     {REGION}")
print(f"Gateway ID: {GATEWAY_ID}")
print(f"VPC ID:     {VPC_USW2_ID}")

In [ ]:
CERT_ARN = input("ACM public certificate ARN: ").strip()
DOMAIN = input("Domain name covered by the certificate (e.g., api.internal.yourcompany.com): ").strip()

assert CERT_ARN.startswith("arn:aws:acm:"), "Invalid certificate ARN"
assert not DOMAIN.startswith("http"), "Domain should not include http:// or https://"
assert "." in DOMAIN, "Domain must contain at least one dot"
assert " " not in DOMAIN, "Domain must not contain whitespace"

print(f"Cert ARN: {CERT_ARN}")
print(f"Domain:   {DOMAIN}")

## 2단계: EKS에 MCP 서버 배포

이 CDK 스택은 다음을 배포합니다.
- FastMCP를 Kubernetes Deployment로 실행하는 **MCP 서버 두 개**(각 서버에 ClusterIP Service 구성)
- 경로 기반 라우팅을 사용하는 **Ingress 리소스**: `/mcp-server/*` → 포트 8000, `/stock-mcp/*` → 포트 8001
- VPC와 연결되고 이름이 `<DOMAIN>`인 **Route 53 프라이빗 호스팅 영역**(처음에는 비어 있으며, NGINX NLB가 프로비저닝되면 Notebook에서 해당 NLB를 가리키는 Alias 레코드를 추가함)

Shared EKS Cluster(단일 내부 NLB 뒤의 NGINX Ingress Controller 및 ACM 인증서 포함)는 별도로 배포됩니다. 단일 NLB가 NGINX 경로 기반 라우팅을 통해 두 MCP 서버를 모두 제공합니다.

In [ ]:
# # NGINX Ingress Controller가 포함된 공유 EKS 클러스터 배포(이미 배포한 경우 건너뛰기)
# # --exclusively는 스택 간 업데이트 문제를 방지함
# !ACCOUNT_A_ID={ACCOUNT_A_ID} cdk deploy SharedEksCluster \
#     -c publicCertArn={CERT_ARN} \
#     --profile {ACCOUNT_A_PROFILE} \
#     --require-approval never \
#     --outputs-file eks-cluster-outputs.json \
#     --exclusively

In [ ]:
# EKS에 MCP 서버 배포(ClusterIP Service + Ingress 리소스 + 프라이빗 호스팅 영역)
# CDK가 McpEks 스택을 합성하려면 publicCertArn이 필요함(인증서 검사로 제어됨)
!ACCOUNT_A_ID={ACCOUNT_A_ID} cdk deploy McpEks \
    -c "publicCertArn={CERT_ARN}" \
    -c "privateDomain={DOMAIN}" \
    --profile {ACCOUNT_A_PROFILE} \
    --require-approval never \
    --outputs-file eks-mcp-outputs.json

In [ ]:
# CDK 출력 읽기(배포 시 프라이빗 호스팅 영역이 생성되며, 아래에서 NLB DNS를 입력함)
with open("eks-mcp-outputs.json") as f:
    eks_mcp_outputs = json.load(f)["McpEks"]

PRIVATE_ZONE_ID = eks_mcp_outputs["PrivateZoneId"]
PRIVATE_DOMAIN = eks_mcp_outputs["PrivateDomain"]
print(f"Private hosted zone: {PRIVATE_DOMAIN}  (zone ID: {PRIVATE_ZONE_ID})")

# NGINX Ingress NLB 검색
print("\nWaiting for NGINX Ingress NLB to be provisioned...")

elbv2_client = session.client("elbv2")
ec2_client = session.client("ec2")
route53_client = session.client("route53")


def _nlb_has_healthy_target(nlb_arn):
    """이 NLB의 target group 중 하나라도 정상 target을 포함하면 True를 반환합니다.

    이 검사가 필요한 이유: NGINX Ingress Service를 재배포하면 AWS Load Balancer
    Controller가 오래된 pod IP를 가리키는 고립된 NLB를 남길 수 있습니다. 두 NLB가
    모두 이름 필터에 일치해도 실제 pod로 라우팅되는 것은 하나뿐입니다. 우연히 첫 번째
    항목을 선택하면 비정상 pod로 라우팅되어 gateway target 호출 시
    "Connection closed by peer"가 표시될 수 있습니다.
    """
    tgs = elbv2_client.describe_target_groups(LoadBalancerArn=nlb_arn)["TargetGroups"]
    for tg in tgs:
        health = elbv2_client.describe_target_health(TargetGroupArn=tg["TargetGroupArn"])["TargetHealthDescriptions"]
        if any(t["TargetHealth"]["State"] == "healthy" for t in health):
            return True
    return False


NLB_DNS = None
NLB_HOSTED_ZONE_ID = None
NLB_SG_ID = None
for attempt in range(20):
    nlbs = elbv2_client.describe_load_balancers()["LoadBalancers"]
    # AWS LB Controller는 NGINX NLB의 이름을 "k8s-ingressn-..." 형식으로 지정함(ingress-nginx에서 잘린 이름)
    nginx_nlbs = [
        n
        for n in nlbs
        if n.get("VpcId") == VPC_USW2_ID
        and n["Scheme"] == "internal"
        and n["Type"] == "network"
        and "k8s-ingressn" in n.get("LoadBalancerName", "").lower()
    ]
    # 정상 대상이 하나 이상 있는 NLB를 우선하여 고립된 NLB를 건너뜀
    healthy = [n for n in nginx_nlbs if _nlb_has_healthy_target(n["LoadBalancerArn"])]
    chosen = healthy[0] if healthy else (nginx_nlbs[0] if nginx_nlbs else None)
    if chosen and (healthy or attempt >= 5):
        # 정상 NLB가 있거나 충분히 기다렸으면 유일한 후보를 선택함
        nlb = chosen
        NLB_DNS = nlb["DNSName"]
        NLB_HOSTED_ZONE_ID = nlb["CanonicalHostedZoneId"]
        NLB_SG_ID = nlb["SecurityGroups"][0] if nlb.get("SecurityGroups") else None
        if len(nginx_nlbs) > 1:
            stale = [n["LoadBalancerName"] for n in nginx_nlbs if n is not nlb]
            print(
                f"WARNING: Multiple matching NLBs found; chose {nlb['LoadBalancerName']}. "
                f"Stale (orphaned by AWS LB Controller): {stale}"
            )
        break
    print(f"  Waiting... (attempt {attempt + 1}/20)")
    time.sleep(15)

assert NLB_DNS, "NGINX Ingress NLB not found. Check if the NGINX Ingress Controller is running."

print(f"\nNLB DNS: {NLB_DNS}")
print(f"NLB SG:  {NLB_SG_ID}")

# VPC CIDR에서 들어오는 443 포트 트래픽을 허용하도록 NLB SG 개방
if NLB_SG_ID:
    try:
        ec2_client.authorize_security_group_ingress(
            GroupId=NLB_SG_ID,
            IpPermissions=[
                {
                    "IpProtocol": "tcp",
                    "FromPort": 443,
                    "ToPort": 443,
                    "IpRanges": [{"CidrIp": "10.0.0.0/16", "Description": "Allow TLS from VPC"}],
                }
            ],
        )
        print(f"Added inbound rule: {NLB_SG_ID} <- TCP 443 from 10.0.0.0/16")
    except ec2_client.exceptions.ClientError as e:
        if "InvalidPermission.Duplicate" in str(e):
            print(f"Inbound rule already exists on {NLB_SG_ID}")
        else:
            raise

# NLB를 가리키는 Alias A 레코드를 프라이빗 호스팅 영역에 UPSERT
route53_client.change_resource_record_sets(
    HostedZoneId=PRIVATE_ZONE_ID,
    ChangeBatch={
        "Changes": [
            {
                "Action": "UPSERT",
                "ResourceRecordSet": {
                    "Name": PRIVATE_DOMAIN,
                    "Type": "A",
                    "AliasTarget": {
                        "HostedZoneId": NLB_HOSTED_ZONE_ID,
                        "DNSName": NLB_DNS,
                        "EvaluateTargetHealth": False,
                    },
                },
            }
        ]
    },
)
print(f"\nUPSERT-ed Alias record: {PRIVATE_DOMAIN} -> {NLB_DNS}")
print("Inside the VPC, the private domain now resolves to the NLB's private IPs.")

## 3단계: AgentCore Gateway 대상 생성

[관리형 VPC 리소스](../01-managed-vpc-resource/)를 사용하여 Gateway 대상을 생성합니다.

- **대상 URL** (`https://{DOMAIN}/mcp-server/mcp`) — 프라이빗 호스팅 영역을 통해 NLB의 프라이빗 IP로 해석됩니다. NGINX Ingress는 Pod로 전달하기 전에 `/mcp-server/mcp`를 `/mcp`로 다시 작성합니다.
- **`managedVpcResource`** — Resource Gateway ENI가 포트 443을 통해 NLB에 접근할 수 있도록 VPC, 서브넷 및 NLB의 보안 그룹을 지정합니다.

두 MCP 서버는 경로 기반 라우팅을 통해 동일한 NLB를 공유합니다. AgentCore Gateway의 Resource Gateway가 Private DNS를 통해 `{DOMAIN}`을 해석하면 NGINX가 경로에 따라 요청을 전달합니다. `routingDomain`은 필요하지 않습니다.

> **보안 그룹:** Resource Gateway ENI가 포트 443을 통해 NLB에 접근할 수 있도록 NLB의 보안 그룹을 `securityGroupIds`에 전달합니다.

In [ ]:
TARGET_ENDPOINT = f"https://{DOMAIN}/mcp-server/mcp"

print(f"Target endpoint: {TARGET_ENDPOINT}  (resolves via Private DNS inside the VPC)")

managed_vpc_resource_config = {
    "vpcIdentifier": VPC_USW2_ID,
    "subnetIds": VPC_USW2_PRIVATE_SUBNETS,
    "endpointIpAddressType": "IPV4",
}
if NLB_SG_ID:
    managed_vpc_resource_config["securityGroupIds"] = [NLB_SG_ID]

response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="eks-mcp-server",
    description="MCP server on EKS via NGINX Ingress and managed VPC egress (Private DNS)",
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": TARGET_ENDPOINT,
                # 대상 간 리소스 URI가 충돌하면 낮은 값이 우선함(8단계 참조)
                "resourcePriority": 10,
            }
        }
    },
    privateEndpoint={
        "managedVpcResource": managed_vpc_resource_config,
    },
)

TARGET_ID = response["targetId"]
print(f"\nTarget ID: {TARGET_ID}")
print(f"Status:    {response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nTarget is active!")
        print(f"  Managed resources: {target.get('privateEndpoint', {})}")
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

## 4단계: AgentCore Gateway를 통해 MCP 서버 호출

Cognito에서 액세스 토큰을 가져온 다음 Gateway를 통해 MCP 서버의 도구를 호출합니다.

In [ ]:
token_response = get_token(
    token_endpoint_url=TOKEN_ENDPOINT_URL,
    client_id=USER_POOL_CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope_string=OAUTH_SCOPES.replace(",", " "),
)
ACCESS_TOKEN = token_response["access_token"]
print(f"Access token obtained (expires in {token_response['expires_in']}s)")

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
}


def _print_req_ids(resp):
    """서비스 로그와 대조할 AgentCore Gateway request/trace ID를 출력합니다."""
    req_id = resp.headers.get("x-amzn-RequestId") or resp.headers.get("x-amz-request-id")
    trace_id = resp.headers.get("X-Amzn-Trace-Id") or resp.headers.get("x-amzn-trace-id")
    print(f"  x-amzn-RequestId: {req_id}")
    if trace_id:
        print(f"  X-Amzn-Trace-Id:  {trace_id}")


# 사용 가능한 도구 나열
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
)
print("Available tools:")
_print_req_ids(response)
print(json.dumps(response.json(), indent=2))

In [ ]:
# 프라이빗 MCP 서버의 "add" 도구 호출
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": "eks-mcp-server___add",
            "arguments": {"a": 5, "b": 3},
        },
        "id": 2,
    },
)
print("Result of add(5, 3):")
_print_req_ids(response)
print(json.dumps(response.json(), indent=2))

## 5단계(선택 사항): 주가 MCP 서버 연결

CDK 스택은 두 번째 MCP 서버인 **모의 주가 서버**를 배포했습니다. 이 서버는 다른 경로(`/stock-mcp/mcp`)를 사용하여 동일한 NGINX Ingress NLB를 통해 라우팅됩니다. 이를 통해 경로 기반 라우팅을 사용하여 단일 로드 밸런서에 **여러 MCP 서버**를 연결하는 방법을 보여 줍니다.

두 번째 대상은 VPC/서브넷/SG 구성이 동일하므로 VPC의 같은 Resource Gateway를 재사용합니다.

In [ ]:
STOCK_TARGET_ENDPOINT = f"https://{DOMAIN}/stock-mcp/mcp"

print(f"Stock target endpoint: {STOCK_TARGET_ENDPOINT}")

# 첫 번째 MCP 서버와 동일한 VPC/서브넷/SG를 사용하고 경로만 다르게 지정
stock_vpc_resource_config = {
    "vpcIdentifier": VPC_USW2_ID,
    "subnetIds": VPC_USW2_PRIVATE_SUBNETS,
    "endpointIpAddressType": "IPV4",
}
if NLB_SG_ID:
    stock_vpc_resource_config["securityGroupIds"] = [NLB_SG_ID]

stock_response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="eks-stock-mcp",
    description="Stock price MCP server on EKS via NGINX Ingress (shared NLB, Private DNS)",
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": STOCK_TARGET_ENDPOINT,
                # mcp-server의 10보다 높으므로 리소스 충돌 시 mcp-server가 우선함(8단계 참조)
                "resourcePriority": 100,
            }
        }
    },
    privateEndpoint={
        "managedVpcResource": stock_vpc_resource_config,
    },
)

STOCK_TARGET_ID = stock_response["targetId"]
print(f"\nStock Target ID: {STOCK_TARGET_ID}")
print(f"Status:          {stock_response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=STOCK_TARGET_ID)
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nStock target is active!")
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

In [ ]:
# 주가 MCP 도구 나열
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
)
print("tools/list:")
_print_req_ids(response)
tools = response.json().get("result", {}).get("tools", [])
stock_tools = [t for t in tools if t["name"].startswith("eks-stock-mcp___")]
print(f"Stock MCP tools ({len(stock_tools)}):")
for t in stock_tools:
    print(f"  {t['name']}: {t.get('description', '')[:80]}")

# 주가 조회
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": "eks-stock-mcp___get_stock_price",
            "arguments": {"symbol": "AAPL"},
        },
        "id": 2,
    },
)
print("\nAAPL stock price:")
_print_req_ids(response)
print(json.dumps(response.json(), indent=2))

## 6단계: Gateway를 통해 프롬프트 사용

MCP **프롬프트**는 서버가 제공하는 매개변수화된 메시지 템플릿입니다. Gateway는 다음 두 가지 MCP 메서드를 전달합니다.

- `prompts/list` — Gateway에 캐시된 카탈로그를 반환합니다. 프롬프트 이름에는 대상 접두사 `{targetName}___{promptName}`가 붙습니다(밑줄 3개, 도구와 동일한 규칙).
- `prompts/get` — 다운스트림 MCP 서버에 **실시간으로** 프록시됩니다. `name` 인수에 `targetName___` 접두사를 포함해야 합니다.

mcp-server는 `order_summary_prompt(orderId)` 프롬프트 하나를 제공합니다.

In [ ]:
# prompts/list
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "prompts/list", "id": 10},
)
print("Available prompts:")
_print_req_ids(response)
print(json.dumps(response.json(), indent=2))

In [ ]:
# prompts/get은 MCP 서버에 실시간으로 프록시됨. 프롬프트 이름 앞에 대상 이름을 붙임.
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "prompts/get",
        "params": {
            "name": "eks-mcp-server___order_summary_prompt",
            "arguments": {"orderId": "123"},
        },
        "id": 11,
    },
)
print("Rendered prompt:")
_print_req_ids(response)
print(json.dumps(response.json(), indent=2))

## 7단계: Gateway를 통해 리소스 사용

MCP **리소스**는 서버가 URI를 통해 제공하는 주소 지정 가능한 콘텐츠입니다. 템플릿화된 리소스는 [RFC 6570 URI 템플릿](https://datatracker.ietf.org/doc/html/rfc6570)을 사용하므로 단일 핸들러가 여러 구체적인 URI를 처리할 수 있습니다. Gateway는 다음 세 가지 메서드를 전달합니다.

- `resources/list` 및 `resources/templates/list` — Gateway의 카탈로그에서 제공됩니다. 리소스 URI는 **원문 그대로** 반환되며 `target___` 접두사가 없습니다.
- `resources/read` — 다운스트림 MCP 서버에 **실시간으로** 프록시됩니다.

mcp-server는 다음 리소스를 제공합니다.
- `orders://catalog` — 정적 리소스
- `orders://{orderId}/details` — 템플릿화된 리소스
- `shared://collision-demo` — stock-mcp도 제공함(8단계에서 `resourcePriority` 시연)

> **보안 경고:** 리소스 URI는 다운스트림 MCP 서버 대상에서 제공하며 Gateway에서 검증하지 않습니다. 악의적이거나 침해된 MCP 서버가 내부 엔드포인트(SSRF) 또는 로컬 파일 시스템 경로(예: `file:///etc/passwd`)를 가리키는 URI를 반환할 수 있습니다. 리소스 URI를 가져오거나 렌더링하기 전에 검증하고 정제하세요.

In [ ]:
# resources/list는 모든 대상의 카탈로그를 병합하고 URI를 원문 그대로 반환함
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "resources/list", "id": 20},
)
print("Available resources:")
_print_req_ids(response)
print(json.dumps(response.json(), indent=2))

In [ ]:
# resources/read는 MCP 서버에 실시간으로 프록시됨(정적 URI)
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "resources/read",
        "params": {"uri": "orders://catalog"},
        "id": 21,
    },
)
print("orders://catalog →")
_print_req_ids(response)
print(json.dumps(response.json(), indent=2))

In [ ]:
# resources/templates/list는 서버가 등록한 RFC 6570 URI 템플릿을 반환함
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "resources/templates/list", "id": 22},
)
print("Resource templates:")
_print_req_ids(response)
print(json.dumps(response.json(), indent=2))

# 템플릿에서 파생된 구체적인 URI를 대상으로 resources/read 호출
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "resources/read",
        "params": {"uri": "orders://123/details"},
        "id": 23,
    },
)
print("\norders://123/details →")
_print_req_ids(response)
print(json.dumps(response.json(), indent=2))

## 8단계: `resourcePriority`를 사용한 다중 대상 충돌 해결

도구와 프롬프트에는 `{targetName}___{name}` 네임스페이스가 자동으로 적용되므로 대상 간 충돌이 발생하지 않습니다. **리소스는 다릅니다.** Gateway는 리소스 URI를 그대로 반환합니다. 두 대상이 동일한 URI를 제공하면 Gateway는 **가장 낮은 `resourcePriority`**를 가진 대상으로 라우팅하여 읽기 요청을 처리합니다(범위 `0–1000`, 기본값 `1000`, 낮은 값 우선).

이 실습에서는 두 MCP 서버가 서로 다른 콘텐츠로 `shared://collision-demo`를 제공합니다.

| 대상 | `resourcePriority` | 반환 콘텐츠 |
|---|---|---|
| `eks-mcp-server` | **10** (낮은 값 → 우선) | `served by mcp-server (resourcePriority=10 wins over stock-mcp=100)` |
| `eks-stock-mcp` | 100 | `served by stock-mcp (resourcePriority=100 — should be shadowed by mcp-server=10)` |

`resources/list`를 호출하면 `shared://collision-demo`가 **두 번** 표시됩니다(대상마다 한 번씩). `resources/read`는 우선순위 값이 더 낮은 대상인 mcp-server로 해석됩니다.

In [ ]:
# resources/list에서 shared://collision-demo가 대상마다 한 번씩 총 두 번 표시됨
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "resources/list", "id": 30},
)
print("resources/list:")
_print_req_ids(response)
collision_entries = [
    r for r in response.json().get("result", {}).get("resources", []) if r.get("uri") == "shared://collision-demo"
]
print(f"shared://collision-demo entries in resources/list: {len(collision_entries)}")
print(json.dumps(collision_entries, indent=2))

In [ ]:
# resources/read 요청을 Gateway가 우선순위 값이 더 낮은 대상으로 라우팅함(mcp-server, priority=10)
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "resources/read",
        "params": {"uri": "shared://collision-demo"},
        "id": 31,
    },
)
print("shared://collision-demo →")
_print_req_ids(response)
print(json.dumps(response.json(), indent=2))
print()
print("Expected: 'served by mcp-server (resourcePriority=10 wins over stock-mcp=100)'")

### 이름 지정 및 충돌 해결 요약

| 기능 | 대상 간 이름 지정 | 충돌 해결 |
|---|---|---|
| 도구 | `targetName___toolName` (밑줄 3개) | 충돌 불가 - 이름에 네임스페이스가 적용됨 |
| 프롬프트 | `targetName___promptName` (밑줄 3개) | 충돌 불가 - 이름에 네임스페이스가 적용됨 |
| 리소스 | 접두사 없이 URI를 원문 그대로 반환 | 대상의 `resourcePriority` 사용(낮은 값 우선, 기본값 1000) |
| 리소스 템플릿 | 접두사 없이 URI 템플릿을 원문 그대로 반환 | `resourcePriority`를 따름 |

## 정리

1. Gateway 대상 삭제
2. CDK 스택 삭제

> **참고:** [REST API Notebook 실습](./api-server-gateway-managed.ipynb)을 실행하지 않는 경우에만 Shared EKS Cluster를 삭제하세요.

In [ ]:
# # 1단계: Gateway 대상 삭제
# agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
# print(f"Deleting target: {TARGET_ID}")
# while True:
#     try:
#         t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
#         print(f"  Status: {t['status']}")
#         time.sleep(15)
#     except agentcore.exceptions.ResourceNotFoundException:
#         print("  Target deleted.")
#         break

# # 주가 대상 삭제(5단계에서 생성한 경우)
# try:
#     agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=STOCK_TARGET_ID)
#     print(f"Deleting stock target: {STOCK_TARGET_ID}")
#     while True:
#         try:
#             t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=STOCK_TARGET_ID)
#             print(f"  Status: {t['status']}")
#             time.sleep(15)
#         except agentcore.exceptions.ResourceNotFoundException:
#             print("  Stock target deleted.")
#             break
# except NameError:
#     pass  # Stock target was not created (Step 5 skipped)

# # 프라이빗 호스팅 영역에서 Alias 레코드 삭제
# # (레코드가 남아 있는 호스팅 영역은 CDK에서 삭제할 수 없음)
# route53_client.change_resource_record_sets(
#     HostedZoneId=PRIVATE_ZONE_ID,
#     ChangeBatch={
#         "Changes": [
#             {
#                 "Action": "DELETE",
#                 "ResourceRecordSet": {
#                     "Name": PRIVATE_DOMAIN,
#                     "Type": "A",
#                     "AliasTarget": {
#                         "HostedZoneId": NLB_HOSTED_ZONE_ID,
#                         "DNSName": NLB_DNS,
#                         "EvaluateTargetHealth": False,
#                     },
#                 },
#             }
#         ]
#     },
# )
# print(f"Deleted Alias record: {PRIVATE_DOMAIN} -> {NLB_DNS}")

In [ ]:
# # 2단계: CDK 스택 삭제
# !ACCOUNT_A_ID={ACCOUNT_A_ID} cdk destroy McpEks \
#     -c "publicCertArn={CERT_ARN}" \
#     -c "privateDomain={DOMAIN}" \
#     --profile {ACCOUNT_A_PROFILE} --force

In [ ]:
# # api-server-gateway-managed.ipynb Notebook을 실행하지 않을 경우에만 EKS 클러스터 삭제
# !ACCOUNT_A_ID={ACCOUNT_A_ID} cdk destroy SharedEksCluster \
#     -c publicCertArn={CERT_ARN} \
#     --profile {ACCOUNT_A_PROFILE} --force \
#     --output cdk-eks.out